In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import re

METHOD:

In this notebook we will start by loading all the data in the order of:

- Percipitation

- River discharge

- Evaporation

- Crop Coefficients

- Areas


Afterwards we will define other parameters before building our water balance model. In the end we will visualize results

PERCIPITATION DATA

In [2]:
percipitation_black = pd.read_csv(r"precipitation data volta\mswep\Black Volta_-2.75_9.55.csv", index_col=0, parse_dates=True)
percipitation_lake = pd.read_csv(r"precipitation data volta\mswep\Lake Volta_0.05_6.45.csv", index_col=0, parse_dates=True)
percipitation_oti = pd.read_csv(r"precipitation data volta\mswep\Oti_0.15_8.45.csv", index_col=0, parse_dates=True)
percipitation_pendjari = pd.read_csv(r"precipitation data volta\mswep\Pendjari_1.15_11.15.csv", index_col=0, parse_dates=True)


In [3]:
percip_blackvolta = percipitation_black /1000
percip_lakevolta = percipitation_lake /1000
percip_oti = percipitation_oti /1000
percip_pendjari = percipitation_pendjari /1000

print(percip_pendjari.head(-5))


            precipitation
time                     
1979-01-01            0.0
1979-01-02            0.0
1979-01-03            0.0
1979-01-04            0.0
1979-01-05            0.0
...                   ...
2017-10-22            0.0
2017-10-23            0.0
2017-10-24            0.0
2017-10-25            0.0
2017-10-26            0.0

[14179 rows x 1 columns]


DISCHARGE DATA

In [4]:
path = r"discharge data volta\discharge in columns.xlsx"
raw = pd.read_excel(path, header=None)

def is_number(x):
    try:
        float(x)
        return True
    except Exception:
        return False

def join_tokens(row, cols):
    toks = [row[c] for c in cols if pd.notna(row[c])]
    s = " ".join(map(str, toks))
    s = re.sub(r"\s+", " ", s).strip()
    return s

records = []
cur_river = None
cur_station = None

for _, row in raw.iterrows():
    # Header rows: first cell is not numeric -> contains river/station fragments
    if not is_number(row[0]):
        cur_river = join_tokens(row, range(0, 6))   # cols 0-5
        cur_station = join_tokens(row, range(6, 11)) # cols 6-10
        continue

    meta = row[11]
    if pd.isna(meta) or cur_river is None:
        continue

    # meta like "2 21977" -> r=2, month=2, year=1977
    parts = str(meta).split()
    if len(parts) != 2:
        continue
    r = int(parts[0])
    my = int(parts[1])
    month = my // 10000
    year = my % 10000

    # values live in cols 0..10 (10 values for row 1/2, up to 11 for row 3)
    vals = pd.to_numeric(row.loc[0:10], errors="coerce").dropna().to_list()
    start_day = (r - 1) * 10 + 1

    for i, v in enumerate(vals):
        day = start_day + i
        # build date; invalid dates (e.g., Feb 30) become NaT and are skipped
        dt = pd.to_datetime({"year": [year], "month": [month], "day": [day]}, errors="coerce")[0]
        if pd.isna(dt):
            continue

        if v == 9999:
            v = np.nan

        records.append({
            "Date": dt,
            "River": cur_river,
            "Station": cur_station,
            "Row": r,
            "Discharge": v
        })

df = pd.DataFrame(records).set_index("Date").sort_index()

# One dataframe per river (each retains Station/Row/Discharge as columns)
river_dfs = {riv: g.copy() for riv, g in df.groupby("River")}



In [5]:
river_names = list(river_dfs.keys())[:]
df_river1, df_river2, df_river3 = (river_dfs[river_names[0]], river_dfs[river_names[1]], river_dfs[river_names[2]])

EVAPORATION DATA

In [6]:
df_evaporation = pd.read_excel('evaporation_daily_design_year.xlsx', index_col=0, parse_dates=True)
df_evaporation = df_evaporation.copy() / 1000 # convert from mm/day to m/day

CROP DATA

In [7]:
crop_data = pd.read_excel("cropdata.xlsx")
print(crop_data.head())
cd = crop_data.copy()
cd.columns = cd.columns.str.strip()

if "crop" not in cd.columns:
    if "Unnamed: 0" in cd.columns:
        cd = cd.rename(columns={"Unnamed: 0": "crop"})
    else:
        cd = cd.rename(columns={cd.columns[0]: "crop"})

cd["crop"] = cd["crop"].astype(str).str.strip()

rename_map = {
    # fix typos here if your file has them
    "Kc end": "Kc_end",
    "Lc_initial (days)": "Lc_initial_days",
    "Lc_develop (days)": "Lc_develop_days",
    "Lc_mid (days)": "Lc_mid_days",
    "Lc_end (days)": "Lc_end_days",
    "start month": "start_month",
    "Kc_intitial": "Kc_initial",           # your earlier typo
    "Lc_intitial (days)": "Lc_initial_days" # your earlier typo
}
cd = cd.rename(columns={k: v for k, v in rename_map.items() if k in cd.columns})

month_map = {m: i for i, m in enumerate(
    ["January","February","March","April","May","June","July","August","September","October","November","December"], start=1
)}
cd["start_month_num"] = cd["start_month"].map(month_map)
# ---------- 2) Create an MM-DD index (365 days, no leap day) ----------
base = pd.date_range("2001-01-01", "2001-12-31", freq="D")  # 2001 is non-leap
md_index = base.strftime("%m-%d")  # strings like '01-01' ... '12-31'

# ---------- 3) Build daily Kc curve on MM-DD index ----------
def kc_design_md_for_crop(row, kc_offseason=0.0):
    # Start with offseason everywhere
    kc = pd.Series(kc_offseason, index=md_index, dtype=float)

    kc_ini = float(row["Kc_initial"])
    kc_mid = float(row["Kc_mid"])
    kc_end = float(row["Kc_end"])

    L_ini = int(row["Lc_initial_days"]) if pd.notna(row["Lc_initial_days"]) else 0
    L_dev = int(row["Lc_develop_days"]) if pd.notna(row["Lc_develop_days"]) else 0
    L_mid = int(row["Lc_mid_days"]) if pd.notna(row["Lc_mid_days"]) else 0
    L_end = int(row["Lc_end_days"]) if pd.notna(row["Lc_end_days"]) else 0

    season_len = L_ini + L_dev + L_mid + L_end
    if season_len <= 0:
        return kc

    start_month = int(row["start_month_num"])
    start_date = pd.Timestamp(year=2001, month=start_month, day=1)

    # Create the season dates in the base non-leap year, and wrap if needed
    season_dates = pd.date_range(start_date, periods=season_len, freq="D")

    part1 = np.full(L_ini, kc_ini)
    part2 = np.linspace(kc_ini, kc_mid, num=L_dev, endpoint=False) if L_dev > 0 else np.array([])
    part3 = np.full(L_mid, kc_mid)
    part4 = np.linspace(kc_mid, kc_end, num=L_end, endpoint=True) if L_end > 0 else np.array([])
    kc_vals = np.concatenate([part1, part2, part3, part4])

    for d, v in zip(season_dates, kc_vals):
        # wrap to within 2001
        wrapped = base[(d - base[0]).days % len(base)]
        kc[wrapped.strftime("%m-%d")] = float(v)

    return kc

kc_by_crop = {row["crop"]: kc_design_md_for_crop(row, kc_offseason=0.0) for _, row in cd.iterrows()}
kc_df_md = pd.DataFrame(kc_by_crop)  # 365 rows, index='MM-DD'

# ---------- 4) Combine to one overall Kc ----------
weights = {
    "Cassava": 0.33,
    "Rice": 0.11,
    "Maize": 0.39,
    "Yam": 0.17
}
w = pd.Series(weights, dtype=float)
w = w / w.sum()

kc_combined_md = (kc_df_md[w.index] * w).sum(axis=1)
kc_combined_md.name = "Kc_combined"

print(kc_combined_md.head(10)) #coefficients for each day of the year
print("\nLength:", len(kc_combined_md)) 

  Unnamed: 0  Kc_intitial  Kc_mid  Kc_end  Lc_intitial (days)  \
0    Cassava         0.30    0.80    0.30                20.0   
1       Rice         1.05    1.20    0.90                30.0   
2      Maize          NaN    1.15    1.05                20.0   
3        Yam          NaN    0.95    0.95                 NaN   

   Lc_develop (days)  Lc_mid (days)  Lc_end (days)  Lc_total (days)  \
0               40.0           90.0           60.0              210   
1               30.0           60.0           30.0              150   
2               25.0           25.0           10.0               80   
3                NaN            NaN            NaN              300   

  start month  
0    November  
1        June  
2        June  
3    December  
01-01    0.264
01-02    0.264
01-03    0.264
01-04    0.264
01-05    0.264
01-06    0.264
01-07    0.264
01-08    0.264
01-09    0.264
01-10    0.264
Name: Kc_combined, dtype: float64

Length: 365


POLYGON AREAS

In [20]:
percipitation_area = pd.read_csv("area_precipitation_volta.csv", index_col=1)

#area of each catchment in m2
df_Acatch = percipitation_area["area_km2"]
area_blackvolta = df_Acatch["Black_volta"] * 1e6
area_lakevolta = df_Acatch["Lake_volta"] * 1e6
area_oti = df_Acatch["Oti"] * 1e6
area_pendjari = df_Acatch["Pendjari"] * 1e6
##area that is cropland within a catchment in m2
df_Airr = percipitation_area["Crop_area_m2"]
crop_blackvolta = df_Airr["Black_volta"]
crop_lakevolta = df_Airr["Lake_volta"]
crop_oti = df_Airr["Oti"]
crop_pendjari = df_Airr["Pendjari"]
##percentage of catchment area that is cropland
crop_blackvolta_per = crop_blackvolta / area_blackvolta
crop_lakevolta_per = crop_lakevolta / area_lakevolta
crop_oti_per = crop_oti / area_oti
crop_pendjari_per = crop_pendjari / area_pendjari
print(crop_blackvolta_per, crop_lakevolta_per, crop_oti_per, crop_pendjari_per)

0.0862945690343757 0.06416867942302204 0.20988613468441694 0.08923156434315048


SUBBASINS
- irrigation area
- catchment area
- which polygon does it fall into?


In [ ]:
########### LOAD THE SUBBASINS FROM QGIS DATA FROM THE REST HERE ############

#####FOR NOW WE ARE GOING TO MAKE UP A BASIN AND RUN THE MODEL ON IT, TO REPLACE LATER ######

In [21]:

area_catchment = (area_lakevolta * 1e6) * 0.01 #1% of lake volta as convert from km^2 to m^2
area_irrigation = area_catchment * crop_lakevolta_per
print(area_irrigation, area_catchment)

21870000000000.0 340820478099999.94


In [22]:
#parameters for the reservoir design model
Cr = 0.31 #runoff coefficient
k = 10e-5 #seepage coefficient m/day
reservoir_area = np.linspace(10000, 400000, num=10)

NOW THE MODEL IN ORDER TO DECIDE THE MINIMUM SURFACE AREA OF THE RESERVOIR IN ORDER TO ALLOW FOR A MAXIMUM OF 10 DAYS OF NO IRRIGATION (EG. 10 CONSECUTIVE DAYS THAT THERE IS NO WATER IN THE RESERVOIR)

In [23]:
SEC_PER_DAY = 86400.0

def volume(area_m2):
    volume = 0.00857 * (area_m2)**1.4367
    return volume #m3

def Rcatch(Cr, Percip, Acatch):
    Rcatch = Cr * Percip * Acatch
    return Rcatch #runoff from catchment in m3/day

def Pdir(Percip, area):
    Pdir = Percip * area
    return Pdir #direct precipitation on reservoir in m3/day

def E(evap, area):
    E = evap * area
    return E #evaporation from reservoir in m3/day

def seep(k, area, L=3):
    seep = k * area * (volume(area) / area) / L
    return seep #seepage loss from reservoir in m3/day

def Dirr(percip, E, Kc, eff_rain=0.8):
    # Dirr in m/day
    return (Kc * E - eff_rain * percip).clip(lower=0.0) #irrigation demand in m/day 

def abstract(Dirr, Airr):
    abstract = Dirr * Airr
    return abstract #abstraction for irrigation in m3/day

def delta_s(area, percip, Acatch, Dirr, Airr, evap, L=3, Rriver=0.0, Cr=Cr, K=k):
    delta_s = (Rcatch(Cr, percip, Acatch) + Rriver + Pdir(percip, area)) - E(evap, area) - seep(K, area, L) - abstract(Dirr, Airr)
    return delta_s #change in storage in m3/day


In [24]:

class ReservoirDesignModel:
    """
    - Uses precipitation and evaporation/ET0 as full time series aligned by date.
    - Only Kc is a design-year (MM-DD) series mapped onto the full period.
    - Uses your functions: volume, Rcatch, Pdir, E, seep, Dirr, abstract.
    - Constraint: reservoir may have Demand>0 and Supply==0 for <= N consecutive days.
    """

    def __init__(
        self,
        percip_df,
        evap_df,
        kc_combined_md,
        area_catchment,
        area_irrigation,
        Cr=Cr,
        k=k,
        L=3,
        eff_rain=0.8,
        Rriver=0.0,
        how_align="inner",          # "inner" keeps only overlapping dates; "outer" keeps all
        fill_missing=None,          # None, "ffill", "bfill", or a float value
    ):
        self.area_catchment = float(area_catchment)   # m2
        self.area_irrigation = float(area_irrigation) # m2
        self.Cr = float(Cr)
        self.k = float(k)
        self.L = float(L)
        self.eff_rain = float(eff_rain)
        self.Rriver = float(Rriver)  # m3/day (constant; can be time series later)

        # --- 1) Convert inputs to daily Series with unique DatetimeIndex ---
        P = self._to_series(percip_df)
        P = self._ensure_daily_unique(P, how="mean")
        P.name = "percip_m_day"

        Ev = self._to_series(evap_df)
        Ev = self._ensure_daily_unique(Ev, how="mean")
        Ev.name = "evap_m_day"

        # --- 2) Align precipitation & evaporation by actual dates (NO design-year creation) ---
        df = pd.concat([P, Ev], axis=1, join=how_align).sort_index()

        if fill_missing is not None:
            if fill_missing in ("ffill", "bfill"):
                df = df.fillna(method=fill_missing)
            else:
                df = df.fillna(float(fill_missing))

        # If still missing values, drop those days (safe default)
        df = df.dropna()

        # --- 3) Map Kc design-year (MM-DD) onto the aligned date index ---
        kc_md = kc_combined_md
        if isinstance(kc_md, pd.DataFrame):
            kc_md = kc_md.iloc[:, 0]
        if not isinstance(kc_md, pd.Series):
            kc_md = pd.Series(kc_md)

        kc_md = kc_md.copy()
        kc_md.index = kc_md.index.astype(str)  # expect "MM-DD"
        df["Kc"] = self._map_md_to_dates(kc_md, df.index)

        self.df = df

    # ---------- helpers ----------
    def _to_series(self, x):
        """Accept Series or DataFrame; return a single Series."""
        if isinstance(x, pd.Series):
            return x
        if isinstance(x, pd.DataFrame):
            # if multiple columns (e.g., multiple pixels/stations), average across columns
            if x.shape[1] > 1:
                return x.mean(axis=1)
            return x.iloc[:, 0]
        raise TypeError("Expected pandas Series or DataFrame.")

    def _ensure_daily_unique(self, s, how="mean"):
        """
        Make sure we have a unique daily DatetimeIndex.
        Handles timestamps, duplicates, etc.
        """
        s = s.copy()
        if not isinstance(s.index, pd.DatetimeIndex):
            s.index = pd.to_datetime(s.index, errors="coerce")
        s = s[~s.index.isna()]
        s.index = s.index.normalize()
        s = s.sort_index()

        if s.index.has_duplicates:
            if how == "sum":
                s = s.groupby(level=0).sum()
            else:
                s = s.groupby(level=0).mean()

        return s.astype(float)

    def _map_md_to_dates(self, md_series, target_dates):
        """
        Map design-year Kc indexed by MM-DD to datetime index.
        Handles leap day by mapping 02-29 -> 02-28 if missing.
        """
        # Ensure md_series index unique
        if not md_series.index.is_unique:
            md_series = md_series.groupby(level=0).mean()

        md = pd.Index(target_dates.strftime("%m-%d"))
        out = md.map(md_series)

        # handle leap day
        if pd.isna(out).any():
            leap_mask = (md == "02-29") & pd.isna(out)
            if leap_mask.any():
                if "02-29" in md_series.index:
                    out = pd.Series(out)
                    out.loc[leap_mask] = float(md_series.loc["02-29"])
                    out = out.values
                elif "02-28" in md_series.index:
                    out = pd.Series(out)
                    out.loc[leap_mask] = float(md_series.loc["02-28"])
                    out = out.values

        return pd.Series(out, index=target_dates, dtype=float)

    def _Dirr_scalar(self, percip_m_day, evap_m_day, kc):
        """
        Your Dirr uses pandas clip; wrap scalars so we reuse your function unchanged.
        """
        sP = pd.Series([percip_m_day])
        sE = pd.Series([evap_m_day])
        sK = pd.Series([kc])
        return float(Dirr(sP, sE, sK, eff_rain=self.eff_rain).iloc[0])  # m/day

    # ---------- simulation ----------
    def simulate(self, area_m2, S0_frac=1.0):
        area_m2 = float(area_m2)
        cap_m3 = float(volume(area_m2))
        S = cap_m3 * float(S0_frac)

        n = len(self.df)
        storage = np.zeros(n)
        supply = np.zeros(n)
        demand = np.zeros(n)
        unmet  = np.zeros(n)

        for i, (_, row) in enumerate(self.df.iterrows()):
            P  = float(row["percip_m_day"])  # m/day
            Ev = float(row["evap_m_day"])    # m/day
            Kc = float(row["Kc"])            # -

            # irrigation demand (m3/day)
            dirr_m = self._Dirr_scalar(P, Ev, Kc)
            Dm3 = float(abstract(dirr_m, self.area_irrigation))
            demand[i] = Dm3

            # inflows (m3/day)
            inflow = float(Rcatch(self.Cr, P, self.area_catchment) + self.Rriver + Pdir(P, area_m2))

            # losses (m3/day)
            loss = float(E(Ev, area_m2) + seep(self.k, area_m2, self.L))

            # available water after inflow/loss
            available = S + inflow - loss
            if available < 0:
                available = 0.0

            # supply limited by availability
            Sm3 = min(Dm3, available)
            supply[i] = Sm3
            unmet[i]  = Dm3 - Sm3

            # update storage, cap at capacity
            S = available - Sm3
            if S > cap_m3:
                S = cap_m3
            storage[i] = S

        return pd.DataFrame(
            index=self.df.index,
            data={
                "percip_m_day": self.df["percip_m_day"].values,
                "evap_m_day": self.df["evap_m_day"].values,
                "Kc": self.df["Kc"].values,
                "Demand_m3_day": demand,
                "Supply_m3_day": supply,
                "Unmet_m3_day": unmet,
                "Storage_m3": storage,
                "Capacity_m3": cap_m3,
                "Reservoir_area_m2": area_m2,
            }
        )

    def max_consecutive_no_supply_days(self, sim_df):
        no_supply = (sim_df["Demand_m3_day"] > 0) & (sim_df["Supply_m3_day"] <= 0)
        max_run = 0
        run = 0
        for v in no_supply.to_numpy():
            if v:
                run += 1
                max_run = max(max_run, run)
            else:
                run = 0
        return int(max_run)

    def find_min_area(self, area_candidates_m2, max_empty_consecutive_days=10, S0_frac=1.0):
        results = []
        for area_m2 in area_candidates_m2:
            sim = self.simulate(area_m2=area_m2, S0_frac=S0_frac)
            max_run = self.max_consecutive_no_supply_days(sim)

            results.append(
                {
                    "area_m2": float(area_m2),
                    "area_ha": float(area_m2) / 10000.0,
                    "capacity_m3": float(volume(float(area_m2))),
                    "max_consec_no_supply_days": max_run,
                    "total_unmet_m3": float(sim["Unmet_m3_day"].sum()),
                    "n_days": int(len(sim)),
                    "start": sim.index.min(),
                    "end": sim.index.max(),
                }
            )

            if max_run <= max_empty_consecutive_days:
                return float(area_m2), sim, pd.DataFrame(results)

        return None, None, pd.DataFrame(results)

In [25]:
model = ReservoirDesignModel(
    percip_df=percip_lakevolta,     # daily 1979-2017, m/day
    evap_df=df_evaporation,         # daily 1979-2017, m/day (NOT mapped)
    kc_combined_md=kc_combined_md,  # design-year MM-DD
    area_catchment=area_catchment,
    area_irrigation=area_irrigation,
    Cr=Cr,
    k=k,
    L=3,
    eff_rain=0.8,
    Rriver=0.0,
    how_align="inner",              # keep overlapping dates only
    fill_missing=None               # or "ffill" if you prefer
)

best_area_m2, sim_best, summary = model.find_min_area(
    area_candidates_m2=reservoir_area,
    max_empty_consecutive_days=10,
    S0_frac=1.0
)

display(summary)

if best_area_m2 is None:
    print("No feasible area found in the candidate set.")
else:
    print(f"Best area: {best_area_m2:.0f} m² ({best_area_m2/10000:.2f} ha)")
    print(f"Capacity: {volume(best_area_m2):.0f} m³")
    print(f"Max consecutive no-supply days: {model.max_consecutive_no_supply_days(sim_best)}")

    sim_best[["Storage_m3"]].plot(title="Reservoir storage (m³)")
    sim_best[["Demand_m3_day","Supply_m3_day","Unmet_m3_day"]].rolling(7).mean().plot(
        title="7-day mean demand/supply/unmet (m³/day)"
    )

,area_m2,area_ha,capacity_m3,max_consec_no_supply_days,total_unmet_m3,n_days,start,end
0,10000.000000,1.000000,4783.886001,86,4.049214e+14,14184,1979-01-01,2017-10-31
1,53333.333333,5.333333,52998.050464,86,4.049213e+14,14184,1979-01-01,2017-10-31
2,96666.666667,9.666667,124545.465113,86,4.049211e+14,14184,1979-01-01,2017-10-31
3,140000.000000,14.000000,212042.526769,86,4.049209e+14,14184,1979-01-01,2017-10-31
4,183333.333333,18.333333,312377.533372,86,4.049206e+14,14184,1979-01-01,2017-10-31
5,226666.666667,22.666667,423707.615791,86,4.049204e+14,14184,1979-01-01,2017-10-31
6,270000.000000,27.000000,544779.983117,86,4.049201e+14,14184,1979-01-01,2017-10-31
7,313333.333333,31.333333,674673.267558,86,4.049198e+14,14184,1979-01-01,2017-10-31
8,356666.666667,35.666667,812674.043302,86,4.049194e+14,14184,1979-01-01,2017-10-31
9,400000.000000,40.000000,958209.365818,86,4.049191e+14,14184,1979-01-01,2017-10-31


No feasible area found in the candidate set.
